# Access, projects and run monitoring

Kernel: **alvheim-tiein (study .venv)** (`studies/alvheim-tiein/.venv`; `uv sync` there first). The repo's own environment lacks pandas and the study package.

Runs that are compared belong to a **project** (`tiein project new`): a type, an
environment, a baseline case and the cases compared against it, including runs made
elsewhere (`tiein project add-run`). Projects are optional: one-off runs use `tiein run`.
This notebook only reads; runs are started deliberately with `tiein submit <case>`, which
refuses more than the project's run limit (at most 50 at once).

In [ ]:
import pandas as pd
import plotly.express as px

from alvheim_tiein import report
from alvheim_tiein.client import AutogrationClient, step_results_frame
from alvheim_tiein.config import Settings
from alvheim_tiein.project import RunLog, Workspace

settings = Settings.from_env()
ws = Workspace(settings.data_dir)
api = AutogrationClient.from_env(settings)
print(settings.api_url, "| auth:", settings.resolved_auth())
api.me()

## Projects

In [ ]:
listing = []
for slug in ws.slugs():
    p, _, log = ws.load(slug)
    statuses = [r["status"] for r in log.rows(p)]
    listing.append({"project": slug, "current": slug == ws.current(), "name": p.name,
                    "type": p.type.value, "baseline": p.baseline_case.name,
                    "cases": len(p.cases), "runs": len(statuses),
                    "completed": statuses.count("Completed")})
pd.DataFrame(listing)

In [ ]:
PROJECT = None  # project slug; None = the current one (`tiein project use <slug>`)
data = report.load_project(ws, PROJECT)
p = data.project
api = AutogrationClient.from_env(settings.for_environment(p.environment))  # the project's runs live there
print(f"{p.name} [{data.slug}, {p.type.value}, {p.environment or 'default'}] model {p.model_id}")
if p.switch:
    print(f"switch: {p.switch.label} ({', '.join(w.name for w in p.switch.wells)}) off from {p.switch.off_from}")
print(f"baseline: {p.baseline_case.name}; booking groups: {', '.join(p.bookings.groups)}")
data.runs

## Run status

Same as `tiein status` (VS Code task **Study: status (watch)**), without updating the run log.

In [ ]:
log = RunLog(data.paths.runs)
for case, entry in log.cases.items():
    role = " (baseline)" if case == p.baseline_case.name else " (imported)" if entry.get("imported") else ""
    if entry.get("assumption_group_id") and not entry.get("imported"):
        progress = api.group_progress(entry["assumption_group_id"]).summary()
    else:
        progress = ", ".join(t["status"] for t in entry["tasks"].values())
    print(f"{case + role:<32} {progress}")

### Tracked variables of a run

The project's `tracked_variables` and, with `bookings.track_applied`, the booking limits
the model applies are recorded at every solver step, so constraint pressure is visible
while the case runs.

In [ ]:
CASE, RUN = None, 1  # e.g. "baseline"; RUN numbers the runs of a sweep
if CASE:
    task_id = data.runs.loc[data.runs["case"] == CASE, "task_id"].iloc[RUN - 1]
    steps = step_results_frame(api.step_results(task_id))
    steps["short"] = steps["variable"].str.replace(r"^resolve\.module\[0\]\.MOD\[\{\w+\}\]\.", "", regex=True)
    px.line(steps, x="time", y="value", color="short", title=f"{CASE} run {RUN}").show()

## Registered models and simulations on the asset

In [ ]:
ASSET = p.asset
models = pd.DataFrame(api.registered_models(asset=ASSET))
models[["task_id", "user_specified_name", "created_at"]] if not models.empty else models

In [ ]:
sims = pd.DataFrame(api.task_page(asset=ASSET, task_type="petex_compute", limit=200)["items"])
if sims.empty:
    print("no simulations on", ASSET)
else:
    display(sims.groupby(["user_specified_name", "status"]).size().unstack(fill_value=0))

## Explore the project's model

Equipment, RESOLVE settings and the optimiser's objective, controls and constraints. The
booking levers of each well (gas-lift injection, production choke dP) are listed by
`tiein project levers`.

In [ ]:
MODEL_ID = p.model_id
display(api.browse(MODEL_ID, search="FL1"))
pd.DataFrame(api.resolve_settings(MODEL_ID)).head(50)